In [13]:
%pip install -q "langchain>=1.0.0" langchain-openai langchain-anthropic langchain-google-genai langchain-text-splitters tiktoken python-dotenv langchain-community faiss-cpu

Note: you may need to restart the kernel to use updated packages.


In [1]:
import os

import langchain
import langchain_core
import langchain_openai

from dotenv import load_dotenv


# .env 로드
load_dotenv()


# API Key 확인
print(
    "[OK] OpenAI API 키 설정됨"
    if os.getenv("OPENAI_API_KEY")
    else "[ERROR] OpenAI API 키 미설정"
)


# LangChain 버전 확인
print(f"LangChain: {langchain.__version__}")
print(f"LangChain Core: {langchain_core.__version__}")
print(f"LangChain OpenAI: {langchain_openai.__version__}")

[OK] OpenAI API 키 설정됨
LangChain: 1.4.0
LangChain Core: 1.6.3
LangChain OpenAI: 1.6.2


### 순차 체인

In [2]:
from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

llm = init_chat_model("gpt-4o-mini", temperature=0.2, max_tokens=2048)

translate_prompt = ChatPromptTemplate.from_template(
    "다음 한국어를 영어로 번역하세요: {korean_word}"
)

explain_prompt = ChatPromptTemplate.from_template(
    "다음 영어 단어를 한국어로 자세히 설명하세요: {english_word}"
)


## 순차 체인
chain1 = translate_prompt | llm | StrOutputParser()

chain2 = (
    {"english_word" : chain1}
    | explain_prompt
    | llm
    | StrOutputParser()
)

result = chain2.invoke({"korean_word": "인공지능"})
print(result)

"인공지능"은 영어로 "artificial intelligence"라고 하며, 이는 인간의 지능을 모방하거나 재현하는 컴퓨터 시스템이나 프로그램을 의미합니다. 인공지능은 기계가 학습, 추론, 문제 해결, 이해, 언어 처리 등과 같은 인간의 인지적 기능을 수행할 수 있도록 하는 기술입니다.

인공지능은 크게 두 가지로 나눌 수 있습니다. 첫째는 "약한 인공지능"으로, 특정 작업을 수행하는 데 특화된 시스템입니다. 예를 들어, 음성 인식 소프트웨어나 추천 알고리즘 등이 이에 해당합니다. 둘째는 "강한 인공지능"으로, 인간과 유사한 수준의 지능을 갖춘 시스템을 의미하지만, 현재로서는 이론적인 개념에 가까운 상태입니다.

인공지능 기술은 머신러닝, 딥러닝, 자연어 처리(NLP), 컴퓨터 비전 등 다양한 분야에서 발전하고 있으며, 의료, 금융, 자율주행차, 고객 서비스 등 여러 산업에 응용되고 있습니다. 인공지능의 발전은 우리의 생활 방식을 변화시키고 있으며, 많은 가능성과 함께 윤리적, 사회적 문제도 동반하고 있습니다.


### 다단계 순차 체인

In [ ]:
from langchain_core.runnables import RunnablePassthrough

analyze_prompt = ChatPromptTemplate.from_template(
    "다음 주제의 핵심 키워드 3개를 추출하세요: {topic}"
)

outline_prompt = ChatPromptTemplate.from_template(
    """다음 키워드를 바탕으로 글의 개요를 작성하세요:
    키워드: {keywords}
    원본 주제: {topic}
    """
    )

conent_prompt = ChatPromptTemplate.from_template(
    """
    다음 개요를 바탕으로 300자 내외의 글을 작성하세요 :
    개요 : {outline}
    """
)

chain = (
    {"topic" : RunnablePassthrough()} 
    | RunnablePassthrough.assign(keywords = analyze_prompt | llm | StrOutputParser())
    | RunnablePassthrough.assign(outline = outline_prompt | llm | StrOutputParser())
    | conent_prompt
    | llm
    | StrOutputParser()
)

result = chain.invoke("기후 변화와 지속 가능한 발전")
print(result)

기후 변화는 지구의 기온 상승과 극단적인 기상 현상을 초래하며, 이는 인간 활동, 특히 산업화와 도시화로 인해 가속화되고 있습니다. 이러한 변화는 생태계와 생물 다양성에 심각한 영향을 미치고, 농업과 건강 등 사회적, 경제적 측면에서도 부정적인 결과를 초래합니다. 지속 가능한 발전은 경제적, 사회적, 환경적 요소를 통합하여 미래 세대의 필요를 충족시키는 것을 목표로 합니다. 이를 위해 재생 가능 에너지 활용, 에너지 효율성 증대, 지속 가능한 농업 등이 필요합니다. 환경 보호는 기후 변화 완화와 밀접한 관계가 있으며, 개인과 공동체의 적극적인 참여가 요구됩니다. 따라서 기후 변화와 지속 가능성의 상호 연관성을 인식하고, 공동의 노력을 통해 책임 있는 행동을 실천해야 합니다.


### 병렬 체인

In [4]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough

positive_prompt = ChatPromptTemplate.from_template(
    "{topic}의 긍정적인 측면 3가지를 한 단어로 설명하세요."
)

negative_prompt = ChatPromptTemplate.from_template(
    "{topic}의 부정적인 측면 3가지를 한 단어로 설명하세요."
)

neutral_prompt = ChatPromptTemplate.from_template(
    "{topic}에 대한 객관적인 현황을 한 단어로 설명하세요."
)

parallel_chain = RunnableParallel(
    positive=positive_prompt | llm | StrOutputParser(),
    negative=negative_prompt | llm | StrOutputParser(),
    neutral=neutral_prompt | llm | StrOutputParser(),
)

results = parallel_chain.invoke({"topic": "원격 근무"})

print("긍정적 측면 :")
print(results["positive"])
print("\n부정적 측면 :")
print(results["negative"])
print("\n객관적 현황 :")
print(results["neutral"])

# 병렬 결과 통합
systhesis_prompt = ChatPromptTemplate.from_template(
    """다음 분석을 종합하여 한문장으로 작성하세요:
        긍정:
        {positive}
        
        단점:
        {negative}
        
        객관적:
        {neutral}
        
        균형 잡힌 결론을 1문장으로 작성하세요.
    """
)

full_chain = (
    parallel_chain
    | systhesis_prompt
    | llm
    | StrOutputParser()
)

results = full_chain.invoke({"topic": "원격 근무"})
print(results)


긍정적 측면 :
유연성, 생산성, 효율성.

부정적 측면 :
고립감, 소통, 집중력.

객관적 현황 :
유연성.
유연성은 생산성과 편리함을 높이는 긍정적인 요소이지만, 고립감과 소통 부족, 집중력 저하라는 단점도 동반될 수 있어 균형 있는 접근이 필요하다.


### 조건부 분기

In [9]:
from langchain_core.runnables import RunnableBranch

delivery_prompt = ChatPromptTemplate.from_template(
    "고객의 배송 문의에 답변하세요: {question}"
)

refund_prompt = ChatPromptTemplate.from_template(
    "고객의 환불 문의에 답변하세요: {question}"
)

default_prompt = ChatPromptTemplate.from_template(
    "상담원 연결로 안내하세요: {question}"
)

def check_request(input_dict):
    question = input_dict["question"]
    
    if "배송" in question:
        return "delivery"
    elif "환불" in question:
        return "refund"
    else:
        return "default"
    
branch_chain = RunnableBranch(
    (lambda x: check_request(x) == "delivery", delivery_prompt | llm | StrOutputParser()),
    (lambda x: check_request(x) == "refund", refund_prompt | llm | StrOutputParser()),
    default_prompt | llm | StrOutputParser()
)

result1 = branch_chain.invoke({"question" : "배송 예정일을 알고싶어요."})
result2 = branch_chain.invoke({"question" : "환불 규정을 알고싶어요."})
result3 = branch_chain.invoke({"question" : "주문을 수정하고싶어요."})

print(result1)
print("------------------")
print(result2)
print("------------------")
print(result3)

안녕하세요! 고객님, 배송 문의 주셔서 감사합니다. 배송 예정일은 주문하신 상품의 종류와 배송 지역에 따라 다를 수 있습니다. 일반적으로 주문 후 2-5일 이내에 배송이 이루어지며, 정확한 배송일은 주문 확인 후 안내해 드리겠습니다. 추가적인 문의사항이 있으시면 언제든지 말씀해 주세요. 감사합니다!
------------------
안녕하세요, 고객님.

환불 규정에 대해 문의 주셔서 감사합니다. 저희 환불 정책은 다음과 같습니다:

1. **환불 요청 기간**: 제품 수령 후 14일 이내에 환불 요청을 하셔야 합니다.
2. **제품 상태**: 환불 요청 시 제품은 사용하지 않은 상태여야 하며, 원래 포장과 함께 반환되어야 합니다.
3. **환불 절차**: 환불을 원하시는 경우, 고객센터에 연락 주시면 자세한 절차를 안내해 드리겠습니다.
4. **환불 처리 시간**: 환불 요청이 승인된 후, 처리 기간은 5-7일 정도 소요될 수 있습니다.

추가적인 질문이 있으시면 언제든지 문의해 주세요. 감사합니다!
------------------
주문 수정에 대한 도움을 드리기 위해 상담원과 연결해 드리겠습니다. 잠시만 기다려 주세요.


### 함수 기반 라우팅

In [11]:
from langchain_core.runnables import RunnableLambda

def route_by_topic(input_dict):
    topic = input_dict.get("topic","").lower()
    
    if "code" in topic or "programming" in topic:
        return "technical"
    elif "business" in topic or "consultant" in topic:
        return "business"
    else:
        return "general"
    
# 주제별 체인 생성
chains = {
    "technical" : ChatPromptTemplate.from_template(
        "기술 전문가로서 답변: {question}") | llm | StrOutputParser(),
    "business" : ChatPromptTemplate.from_template(
        "비즈니스 컨설턴트로서 답변: {question}") | llm | StrOutputParser(),
    "general" : ChatPromptTemplate.from_template(
        "일반적 관점에서 답변: {question}") | llm | StrOutputParser(),
}

def routing_chain(input_dict):
    # 어디로 보낼지 결정
    route = route_by_topic(input_dict)
    # 결정된 곳의 체인을 가져와서 실행
    return chains[route].invoke(input_dict)

router = RunnableLambda(routing_chain)

result = router.invoke({
    "topic" : "LangChain",
    "question" : "효율적인 작성 방법은?"
})

print(result)

    
    

효율적인 작성 방법은 여러 가지 요소를 고려해야 합니다. 다음은 일반적인 관점에서 효율적으로 글을 작성하는 방법입니다:

1. **목적 설정**: 글을 쓰기 전에 작성의 목적을 명확히 하세요. 독자가 누구인지, 어떤 정보를 전달하고 싶은지를 생각합니다.

2. **계획 세우기**: 글의 구조를 미리 계획합니다. 서론, 본론, 결론의 형식을 따르거나, 주제별로 나누어 정리할 수 있습니다.

3. **자료 조사**: 필요한 정보를 미리 조사하고 정리합니다. 신뢰할 수 있는 출처에서 자료를 수집하는 것이 중요합니다.

4. **초안 작성**: 처음부터 완벽하게 쓰려고 하지 말고, 아이디어를 자유롭게 적어보세요. 초안은 수정할 수 있는 기초입니다.

5. **명확한 문장 사용**: 간결하고 명확한 문장을 사용하여 독자가 쉽게 이해할 수 있도록 합니다. 복잡한 표현은 피하는 것이 좋습니다.

6. **적절한 단어 선택**: 주제에 맞는 적절한 어휘를 사용하고, 전문 용어는 필요할 경우에만 사용합니다.

7. **수정 및 편집**: 초안을 작성한 후에는 반드시 수정과 편집을 거쳐야 합니다. 문법, 철자, 문장 구조 등을 점검하고, 내용의 일관성을 확인합니다.

8. **피드백 받기**: 다른 사람에게 글을 읽어보게 하고 피드백을 받는 것도 좋은 방법입니다. 새로운 시각에서의 의견이 도움이 될 수 있습니다.

9. **시간 관리**: 글쓰기 시간을 정해두고, 집중해서 작업하는 것이 중요합니다. 방해 요소를 최소화하고, 정해진 시간 내에 작업을 완료하도록 노력합니다.

10. **연습**: 글쓰기는 연습을 통해 향상됩니다. 꾸준히 글을 쓰고 다양한 스타일을 시도해보세요.

이러한 방법들을 활용하면 보다 효율적으로 글을 작성할 수 있습니다.


### 복합 패턴

In [ ]:
from langchain_core.documents import Document
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

# 검색할 자료
docs= [
    Document(page_content="LangChain은 LLM 애플리케이션을 만들기 위한 프레임워크입니다."),
    Document(page_content="RAG는 외부 문서를 검색한 뒤 그 정보를 LLM에게 제공하는 방식입니다."),
    Document(page_content="RunnableBranch는 조건에 따라 서로 다른 체인을 실행합니다."),
]

# 문장을 숫자로 바꿔주는 도구(벡터화)
embeddings = OpenAIEmbeddings()

# 검색할 수 있도록 벡터화된 문서들을 가지고 있는 FAISS 저장소
# 이 문서들을 이 임베딩 방식으로 정리해서 FAISS 검색창고 하나 만들어줘.
vectorstore = FAISS.from_documents(docs,embeddings)

In [ ]:
# 입력, 출력 str 타입 힌트
def retrieve_context(query: str) -> str:
    # LangChain의 FAISS 연동 클래스가 제공하는 .similarity_search 유사도 검색 기능
    docs = vectorstore.similarity_search(query, k=1)
    
    return "\n".join(
        doc.page_content
        for doc in docs
    )

In [ ]:
rag_chain = (
    RunnableParallel(
        question=RunnablePassthrough(),
        context=RunnableLambda(
            lambda x: retrieve_context(x["question"])
        )
    )

    | ChatPromptTemplate.from_template(
        """
컨텍스트를 참고하여 질문에 답변하세요.

컨텍스트:
{context}

질문:
{question}

답변:
"""
    )

    | llm
    | StrOutputParser()
)

result = rag_chain.invoke({
    "question": "LangChain이란?"
})

print(result)